# Notebook 01 - Data Profiling (M3 Stage 1)

Loads all raw sources, profiles shape/dtypes/nulls/ranges. Identifies issues for cleaning.

In [ ]:
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from src.utils.helpers import setup_logging
setup_logging()
sns.set_theme(style="whitegrid")
FIGURES = "../reports/figures"
os.makedirs(FIGURES, exist_ok=True)
print("Environment ready.")

## 1. Load all raw sources

In [ ]:
from src.loaders.load_air_kaggle import load_city_day, aggregate_to_city_year
from src.loaders.load_safety import load_safety
from src.loaders.load_health import load_health
from src.loaders.load_economy import load_economy
from src.loaders.load_cost_of_living import load_cost_of_living

sources = {}
for name, fn in [('air_daily', load_city_day),('safety', load_safety),('health', load_health),('economy', load_economy),('col', load_cost_of_living)]:
    try:
        sources[name] = fn()
        print(f'{name}: {sources[name].shape}')
    except Exception as e:
        print(f'{name}: {e}')
        sources[name] = pd.DataFrame()
if 'air_daily' in sources and not sources['air_daily'].empty:
    sources['air_annual'] = aggregate_to_city_year(sources['air_daily'])

## 2. Shape, dtypes and missing values

In [ ]:
from src.utils.helpers import missing_summary
for name, df in sources.items():
    if df.empty: continue
    print(f'\n{"="*55}\n  {name}\n{"="*55}')
    print(f'  Shape: {df.shape}')
    print(df.dtypes.to_string())
    ms = missing_summary(df)
    if not ms.empty: print('  Missing:\n', ms.to_string())
    else: print('  No missing values.')

## 3. Numeric summaries

In [ ]:
for name, df in sources.items():
    num = df.select_dtypes('number')
    if num.empty or df.empty: continue
    print(f'\n--- {name} ---')
    display(num.describe().round(2))

## 4. Air quality daily overview

In [ ]:
if not sources.get('air_daily', pd.DataFrame()).empty:
    daily = sources['air_daily'].copy()
    daily['year'] = daily['date'].dt.year
    fig, axes = plt.subplots(1, 3, figsize=(14,4))
    axes[0].hist(daily['pm25'].dropna(), bins=50, color='#3498db', edgecolor='white')
    axes[0].set_xlabel('PM2.5'); axes[0].set_title('PM2.5 Distribution')
    city_counts = daily['city'].value_counts().head(12)
    axes[1].barh(city_counts.index[::-1], city_counts.values[::-1], color='#2ecc71')
    axes[1].set_title('Top 12 Cities by Record Count')
    yearly = daily.groupby('year')['pm25'].mean()
    axes[2].bar(yearly.index, yearly.values, color='#9b59b6')
    axes[2].set_xlabel('Year'); axes[2].set_ylabel('Mean PM2.5')
    axes[2].set_title('Mean PM2.5 by Year')
    plt.tight_layout()
    plt.savefig(f'{FIGURES}/01_air_overview.png', dpi=150, bbox_inches='tight')
    plt.show()

## 5. Coverage heatmap: city x year

In [ ]:
annual = sources.get('air_annual', pd.DataFrame())
if not annual.empty:
    top_cities = annual.groupby('city')['year'].count().nlargest(20).index
    piv = annual[annual['city'].isin(top_cities)].pivot_table(index='city', columns='year', values='pm25_mean', aggfunc='mean')
    fig, ax = plt.subplots(figsize=(12,7))
    sns.heatmap(piv, ax=ax, cmap='YlOrRd', linewidths=0.3, linecolor='white', annot=True, fmt='.0f')
    ax.set_title('Annual PM2.5 Heatmap (top 20 cities)')
    plt.tight_layout()
    plt.savefig(f'{FIGURES}/01_coverage_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()

## Issues log

In [ ]:
print('=== ISSUES FOR CLEANING ===\n')
for name, df in sources.items():
    if df.empty: continue
    ms = missing_summary(df)
    n_dup = df.duplicated().sum()
    issues = []
    if not ms.empty: issues.append(f'Missing: {list(ms.index)}')
    if n_dup > 0: issues.append(f'{n_dup} duplicate rows')
    if issues: print(f'{name}:')
    for i in issues: print(f'  - {i}')
    if not issues: print(f'{name}: OK')